In [1]:
import re, os, sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from pathlib import Path
import shutil

In [2]:
load_dotenv()

True

In [3]:
DATASET_PATH = Path(os.getenv("PROJECT_ROOT_DIR")) / "dataset" / "pypi_pylingual_dataset_merged_pycdc_pylingual.csv"
ROOT_FOR_FILES = Path(os.getenv("ROOT_FOR_FILES"))

In [4]:
final_dataset_df = pd.read_csv(DATASET_PATH)

In [5]:
final_dataset_df.shape

(1637, 8)

In [6]:
def build_file_path(row, root_for_files: Path):
    """
    Build the file path based on dataset and decompiler values.
    """

    file_hash = row["file_hash"]
    file_name = row["file"]
    decompiler = row["decompiler"]
    dataset = row["dataset"]

    if decompiler == "pylingual":
        if dataset == "pylingual":
            return root_for_files / "pylingual_download" / "decompiler_workspace" / file_hash / "decompiler_output" / file_name
        elif dataset == "pypi":
            return root_for_files / "pypi_downloaded" / file_hash / "decompiled_output" / file_name

    elif decompiler == "pycdc":
        if dataset == "pylingual":
            return root_for_files / "pylingual_download" / "decompiler_workspace" / file_hash / "pycdc_decompilation_output" / file_name
        elif dataset == "pypi":
            return root_for_files / "pypi_downloaded" / file_hash / "decompiled_output_pycdc" / file_name

    return None

In [7]:
final_dataset_df["file_path"] = final_dataset_df.apply(
    lambda row: build_file_path(row, ROOT_FOR_FILES),
    axis=1
)

In [8]:
final_dataset_df.dropna(subset=["file_path"], inplace=True)

In [9]:
def extract_filename_from_error_description(error_description: str):
    if not isinstance(error_description, str):
        return None

    # Format 1: File "/path/to/file.py", line N
    m = re.search(r'File\s+"+([^"]+)"+', error_description)
    if m:
        return Path(m.group(1)).name

    # Format 2: Sorry: ... (filename.py, line N)
    m = re.search(r'\(([^(),]+\.py),\s*line\s*\d+\)', error_description)
    if m:
        return Path(m.group(1)).name

    return None


def replace_file_for_invalid_rows(df):
    """
    For rows whose file_path is invalid, extract filename from error_description
    and replace the file column with that filename.
    """
    df = df.copy()

    invalid_mask = ~df["file_path"].map(lambda p: Path(p).exists())

    df.loc[invalid_mask, "file"] = df.loc[invalid_mask, "error_description"].apply(
        extract_filename_from_error_description
    )

    return df

In [10]:
final_dataset_df = replace_file_for_invalid_rows(final_dataset_df)

In [11]:
final_dataset_df["file_path"] = final_dataset_df.apply(
    lambda row: build_file_path(row, ROOT_FOR_FILES),
    axis=1
)

### Template for file path ###

    - decompiler->pylingual:
        - dataset->pylingual: {root_for_files}/pylingual_download/decompiler_workspace/{hash}/decompiler_output/{file}
        - dataset->pypi:  {root_for_files}/pypi_downloaded/{hash}/decompiled_output/{file}
    - decompiler->pycdc:
        - dataset->pylingual: {root_for_files}/pylingual_download/decompiler_workspace/{hash}/pycdc_decompilation_output/{file}
        - dataset->pypi: {root_for_files}/pypi_downloaded/{hash}/decompiled_output_pycdc/{file}

In [12]:

invalid_rows = final_dataset_df[
    ~final_dataset_df["file_path"].map(lambda p: Path(p).exists())
]

In [13]:
invalid_rows.shape

(0, 9)

In [14]:
def normalize_bytecode_version(value):
    try:
        return f"{float(value):.2f}"
    except Exception:
        return str(value).strip()


def move_pylingual_pypi_312_files_safely(df: pd.DataFrame, root_for_files: Path, dry_run: bool = True):
    required_cols = {"file_hash", "file", "decompiler", "dataset", "bytecode_version"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    work_df = df.copy()
    work_df["bytecode_version_norm"] = work_df["bytecode_version"].apply(normalize_bytecode_version)

    mask = (
        (work_df["decompiler"].astype(str).str.strip() == "pylingual") &
        (work_df["dataset"].astype(str).str.strip() == "pypi") &
        (work_df["bytecode_version_norm"] == "3.12")
    )

    candidates = work_df.loc[mask, ["file_hash", "file", "decompiler", "dataset", "bytecode_version"]].copy()

    actions = []

    for _, row in candidates.iterrows():
        file_hash = str(row["file_hash"]).strip()
        file_name = Path(str(row["file"]).strip()).name

        if not file_hash or not file_name:
            actions.append({
                "file_hash": file_hash,
                "file": file_name,
                "status": "skip_missing_hash_or_file",
                "source_path": None,
                "destination_path": None,
            })
            continue

        hash_dir = root_for_files / "pypi_downloaded" / file_hash
        source_path = hash_dir / file_name
        destination_dir = hash_dir / "decompiled_output"
        destination_path = destination_dir / file_name

        if destination_path.exists():
            actions.append({
                "file_hash": file_hash,
                "file": file_name,
                "status": "skip_destination_already_exists",
                "source_path": str(source_path),
                "destination_path": str(destination_path),
            })
            continue

        if not source_path.exists():
            actions.append({
                "file_hash": file_hash,
                "file": file_name,
                "status": "skip_source_missing",
                "source_path": str(source_path),
                "destination_path": str(destination_path),
            })
            continue

        if not source_path.is_file():
            actions.append({
                "file_hash": file_hash,
                "file": file_name,
                "status": "skip_source_not_file",
                "source_path": str(source_path),
                "destination_path": str(destination_path),
            })
            continue

        if not dry_run:
            destination_dir.mkdir(parents=True, exist_ok=True)
            shutil.move(str(source_path), str(destination_path))
            status = "moved"
        else:
            status = "would_move"

        actions.append({
            "file_hash": file_hash,
            "file": file_name,
            "status": status,
            "source_path": str(source_path),
            "destination_path": str(destination_path),
        })

    return pd.DataFrame(actions)

In [15]:
move_log = move_pylingual_pypi_312_files_safely(
    df=invalid_rows,
    root_for_files=ROOT_FOR_FILES,
    dry_run=True
)

print(move_log["status"].value_counts(dropna=False))

status
would_move    97
Name: count, dtype: int64


In [16]:
move_log[move_log["status"] == "would_move"].iloc[0]["destination_path"]

'/home/mxs220189/pylingual_collaboration/pypi_downloaded/0224c3f609f88cdc1fa10132cd0bab7f5f80970e7d8421dc422c3199bc33de5385149f40eddc1d0292627f0004929b6e02c65349ab076fc8df4526d82bea5bb0/decompiled_output/decompiled_Extensions.cpython-312.py'

In [17]:
move_log = move_pylingual_pypi_312_files_safely(
    df=invalid_rows,
    root_for_files=ROOT_FOR_FILES,
    dry_run=False
)

In [15]:
final_dataset_df["file_path"] = final_dataset_df.apply(
    lambda r: build_file_path(r, ROOT_FOR_FILES),
    axis=1
)

invalid_rows_after = final_dataset_df[
    ~final_dataset_df["file_path"].map(lambda p: Path(p).exists())
]

In [16]:
final_dataset_df["bytecode_version"] = final_dataset_df["bytecode_version"].astype(str)

final_dataset_df["bytecode_version"] = final_dataset_df["bytecode_version"].replace({
    "3.1": "3.10"
})

In [34]:
final_dataset_df = pd.read_csv(DATASET_PATH)

In [35]:
final_dataset_df.columns

Index(['file_hash', 'file', 'error_message', 'error_description', 'error',
       'bytecode_version', 'decompiler', 'dataset', 'file_path'],
      dtype='str')

In [30]:
final_dataset_df["decompiler"].isna().sum()

np.int64(0)

In [31]:
invalid_rows_after

,file_hash,file,error_message,error_description,error,bytecode_version,decompiler,dataset,file_path


In [ ]:
group_cols = ["decompiler", "dataset", "bytecode_version"]
min_per_group = 2
random_state = 42

subset_df = (
    final_dataset_df.groupby(group_cols, group_keys=False, dropna=False)
      .apply(lambda g: g.sample(n=min(min_per_group, len(g)), random_state=random_state))
      .sample(frac=1, random_state=random_state)
      .reset_index(drop=True)
)

# subset_df.to_csv("test_subset.csv", index=False)

print(subset_df.groupby(group_cols).size().reset_index(name="count"))

KeyError: 'decompiler'

In [26]:
subset_df = subset_df.sample(frac=1, random_state=random_state).reset_index(drop=True)

In [27]:
subset_df.to_csv(Path(os.getenv("PROJECT_ROOT_DIR")) / "dataset" / "test_subset.csv", index=False)